# HYPER-3, 3 — How big, allocated how, powered for what

Notebook 2 said what the trial identifies. This one turns that into integers: how many
units, split how between four arms, and what the resulting design can and cannot see.

The number that matters arrives in section 5. The pooled question needs 71 units and
HYPER-3 enrolls 400, because the protocol commits to watching each dose **inside each
age stratum**. At closeout that is enough: a stratum-level contrast detects about 4
mmHg, and the harm the case study is about is 5.8. At the *first safety review*, with
three eighths of the information, the same contrast has 69 % power against that harm
and a minimum detectable effect sitting above it — one look is a coin flip, and six
looks tested at 5 % apiece have no error control whatever. That is the problem
notebook 5 exists to solve.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import hyper3 as h
from axiom.core import D, Unit, Unsupported, is_failure
from axiom.design import (
    MDE, Assignment, ClusterDesign, CostPerOutcomeInterval, CostPerOutcomePower, HoldoutTradeoff,
    PowerCurve, PowerResult, SampleSize, cluster_mde, cluster_power, clusters_needed,
    coefficient_mde, coefficient_power, coefficient_sample_size, cost_per_outcome_interval,
    cost_per_outcome_power, design_effect, difference_se, effective_sample_size, holdout_tradeoff,
    match_clusters, max_detectable_cost_per_outcome, mde, power, power_curve, power_from_se,
    sample_size,
)
from axiom.identify import ols

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

# The design numbers, fixed before the trial opened.
SD_SINGLE_VISIT = 8.8       # sd of one unit's change at one visit, from the phase I extension
SD_WINDOW = 6.0             # sd of the mean of four weekly readings — notebook 1 measured 6.02
MEANINGFUL = 4.0            # mmHg: the smallest reduction anyone would change practice for
ALPHA, TARGET_POWER = 0.05, 0.80
ATTRITION = 0.08            # share expected not to reach the primary window
print(f"powering for {MEANINGFUL} {h.OUTCOME_UNIT} at sd {SD_WINDOW} {h.OUTCOME_UNIT}, "
      f"{TARGET_POWER:.0%} power, two-sided {ALPHA}")

## 1. Two arms first

Every number below comes from one formula: the standard error of a difference in means
is `sd / sqrt(n · p · (1 − p))`, and power is the normal tail beyond the critical
value. `sample_size` inverts it exactly, so the power achieved at the returned integer
`n` is at or above the target rather than near it.

In [ ]:
two_arm = sample_size(effect=MEANINGFUL, sd=SD_WINDOW, power=TARGET_POWER, alpha=ALPHA)
assert isinstance(two_arm, SampleSize)
print(f"two-arm n = {two_arm.n} ({two_arm.n_treated} treated / {two_arm.n_control} control), "
      f"se {two_arm.se:.3f}, achieved power {two_arm.power:.3f}")
print(f"the same trial reading one visit instead of the four-week window would need "
      f"n = {sample_size(effect=MEANINGFUL, sd=SD_SINGLE_VISIT, power=TARGET_POWER).n}")
print(f"weekly measurement is therefore worth "
      f"{sample_size(effect=MEANINGFUL, sd=SD_SINGLE_VISIT).n - two_arm.n} units of enrollment")

In [ ]:
grid = np.linspace(0.5, 8.0, 61)
fig = h.figure("Power against effect size, for four candidate arm sizes",
               f"true reduction in SBP ({h.OUTCOME_UNIT})", "power", height=400)
for n_total, colour in ((80, "#c9c9c9"), (120, "#8ab4e8"), (two_arm.n, "#2f7fd1"), (240, "#20456b")):
    curve = power_curve(n_total, SD_WINDOW, tuple(grid), alpha=ALPHA)
    assert isinstance(curve, PowerCurve)
    fig.add_trace(go.Scatter(x=curve.effects, y=curve.powers, mode="lines", name=f"n = {n_total}",
                             line={"color": colour, "width": 2.6}))
fig.add_hline(y=TARGET_POWER, line={"color": "rgba(0,0,0,0.4)", "dash": "dash"},
              annotation_text="80 %")
fig.add_vline(x=MEANINGFUL, line={"color": "#d1483f", "dash": "dot"},
              annotation_text=f"{MEANINGFUL} {h.OUTCOME_UNIT}")
fig

## 2. Four arms, one control

HYPER-3 makes three comparisons against a single control arm, and every unit put in
the control arm improves all three. The classical answer is the **square-root rule**:
with `k` experimental arms sharing one control, the standard error of the average
comparison is minimized at a control share of `sqrt(k) : 1` per arm. For `k = 3` that
is 1.73 : 1 : 1 : 1 — and the 2 : 1 : 1 : 1 blocks the protocol actually uses are
within a fraction of a percent of it.

In [ ]:
N_TOTAL = 400
K = len(h.ARMS) - 1
ratios = np.linspace(0.6, 4.0, 120)
ses = []
for ratio in ratios:
    n_control = N_TOTAL * ratio / (ratio + K)
    n_dose = (N_TOTAL - n_control) / K
    ses.append(SD_WINDOW * float(np.sqrt(1.0 / n_dose + 1.0 / n_control)))
best = float(ratios[int(np.argmin(ses))])
protocol_ratio = h.ALLOCATION["standard_of_care"] / h.ALLOCATION["dose_40"]
print(f"square-root rule says {np.sqrt(K):.3f} : 1; the grid minimum is at {best:.2f} : 1")
print(f"the protocol uses {protocol_ratio:.0f} : 1, costing "
      f"{(np.interp(protocol_ratio, ratios, ses) / min(ses) - 1) * 100:.2f}% of standard error")

fig = h.figure("Control-arm allocation with three doses sharing one control",
               "control units per dose-arm unit", f"se of one dose-vs-control contrast ({h.OUTCOME_UNIT})",
               height=380)
fig.add_trace(go.Scatter(x=ratios, y=ses, mode="lines", line={"color": "#2f7fd1", "width": 2.6},
                         name="standard error", showlegend=False))
fig.add_vline(x=float(np.sqrt(K)), line={"color": "#d1483f", "dash": "dot"},
              annotation_text="√k = 1.73")
fig.add_vline(x=protocol_ratio, line={"color": "#111", "dash": "dash"},
              annotation_text="protocol 2:1")
fig

In [ ]:
n_per_arm = {arm: N_TOTAL * k / h.BLOCK for arm, k in h.ALLOCATION.items()}
rows = []
for arm in h.ARMS[1:]:
    n_pair = n_per_arm[arm] + n_per_arm["standard_of_care"]
    allocation = n_per_arm[arm] / n_pair
    se = difference_se(int(n_pair * (1 - ATTRITION)), sd=SD_WINDOW, allocation=allocation)
    detectable: MDE = mde(int(n_pair * (1 - ATTRITION)), sd=SD_WINDOW, power=TARGET_POWER,
                          allocation=allocation)
    at_meaningful: PowerResult = power_from_se(MEANINGFUL, se, alpha=ALPHA)
    rows.append({"arm": h.ARM_LABEL[arm], "n_dose": int(n_per_arm[arm]),
                 "n_control": int(n_per_arm["standard_of_care"]), "se": se,
                 "mde_80": detectable.effect, "power_at_4mmHg": at_meaningful.power})
plan = pd.DataFrame(rows)
print(f"planned {N_TOTAL} units in 2:1:1:1 blocks, {ATTRITION:.0%} attrition assumed")
print(plan.round(3).to_string(index=False))

## 3. What the ANCOVA coefficient actually costs

The primary analysis is not a two-sample difference; it is the coefficient on the
treatment indicator in a regression that also adjusts for baseline pressure and
stratum. The `coefficient_*` functions take that design standard error directly rather
than re-deriving it from `sd` and `n`, and here we can check the promise against the
trial's realized standard error.

In [ ]:
trial = h.trial(seed=20260821)
final = h.look_frame(trial, h.TRIAL_WEEKS, endpoint="primary")
realized = ols(h.contrast_frame(final, "dose_20"), "change", "treated", h.ancova_covariates(None))
promised = float(plan.loc[plan["arm"] == "20 mg", "se"].iloc[0])
print(f"design promised se {promised:.3f}; the trial realized se {realized.se:.3f} on n = {realized.n}")
print(f"coefficient MDE at the realized se: {coefficient_mde(realized.se, power=TARGET_POWER).effect:.3f}")
print(f"power for {MEANINGFUL} at the realized se: {coefficient_power(MEANINGFUL, realized.se).power:.3f}")
needed = coefficient_sample_size(2.5, realized.se, realized.n, power=TARGET_POWER)
print(f"to detect 2.5 {h.OUTCOME_UNIT} at 80% power this analysis would need n = "
      f"{needed.n if isinstance(needed, SampleSize) else needed}")

ANCOVA beat the plain two-sample formula: adjusting for a unit's own baseline pressure
removes real variance, and the realized standard error came in below what the design
promised. That surplus is why the design's attrition allowance was not the binding
constraint.

## 4. Why the trial is 400 and not 71

The pooled question needs 71 units. HYPER-3 enrolls 400 because the protocol commits to
something the pooled question never asks for: monitoring each dose **inside each age
stratum**. The oldest band is 35 % of the trial, so its 40 mg contrast rests on 28
units against 56, and the minimum detectable effect there is nearly twice the pooled
one. Choosing 400 is choosing to be able to see a stratum-level effect of about 4 mmHg.

In [ ]:
rows = []
for stratum in h.STRATA:
    n_stratum = int((trial.units["stratum"] == stratum).sum())
    n_dose = n_stratum * h.ALLOCATION["dose_40"] / h.BLOCK
    n_control = n_stratum * h.ALLOCATION["standard_of_care"] / h.BLOCK
    n_pair = int((n_dose + n_control) * (1 - ATTRITION))
    allocation = n_dose / (n_dose + n_control)
    detectable = mde(n_pair, sd=SD_WINDOW, power=TARGET_POWER, allocation=allocation)
    se = difference_se(n_pair, sd=SD_WINDOW, allocation=allocation)
    truth = h.intent_to_treat_contrast(40.0, stratum)
    rows.append({"stratum": h.STRATUM_LABEL[stratum], "n_dose": int(n_dose),
                 "n_control": int(n_control), "se": se, "mde_80": detectable.effect,
                 "true_40mg_effect": truth, "power_at_truth": power_from_se(truth, se).power})
strata_plan = pd.DataFrame(rows)
pooled_mde = float(plan.loc[plan["arm"] == "40 mg", "mde_80"].iloc[0])
print(f"pooled 40 mg MDE at 80 % power: {pooled_mde:.2f} {h.OUTCOME_UNIT}")
print(strata_plan.round(3).to_string(index=False))

fig = h.figure("Minimum detectable effect at closeout: pooled versus inside a stratum", "",
               f"MDE at 80 % power ({h.OUTCOME_UNIT})", height=400)
fig.add_trace(go.Bar(x=strata_plan["stratum"], y=strata_plan["mde_80"], name="stratum MDE",
                     marker_color=[h.STRATUM_COLOR[s] for s in h.STRATA]))
fig.add_trace(go.Scatter(x=strata_plan["stratum"], y=strata_plan["true_40mg_effect"].abs(),
                         mode="markers", name="|true effect| at 40 mg",
                         marker={"symbol": "diamond", "size": 13, "color": "#111"}))
fig.add_hline(y=pooled_mde, line={"color": "#2f7fd1", "dash": "dash"},
              annotation_text=f"pooled MDE {pooled_mde:.1f}")
fig

At closeout every stratum's MDE sits below the 40 mg effect it is facing except the
middle band, where the true effect is genuinely small. So the trial as designed *can*
find a stratum-level harm — eventually.

## 5. What an interim costs

A safety boundary does not get to wait for closeout; it has to act on the information
that has accrued. Standard errors scale as `1 / sqrt(information)`, so the minimum
detectable effect at information fraction `t` is the closeout MDE divided by `sqrt(t)`.
HYPER-3's first safety review is at **three eighths** of the safety endpoints — earlier
than that a stratum-level contrast rests on a handful of units per arm and has no normal
approximation worth acting on.

In [ ]:
REVIEWS = (0.375, 0.5, 0.625, 0.75, 0.875, 1.0)
fractions = np.linspace(0.15, 1.0, 86)
fig = h.figure("Detectable effect against information accrued, 40 mg versus control",
               "information fraction", f"MDE at 80 % power ({h.OUTCOME_UNIT})", height=400)
for row, stratum in zip(strata_plan.itertuples(), h.STRATA, strict=True):
    detectable = [coefficient_mde(row.se / float(np.sqrt(t)), power=TARGET_POWER).effect
                  for t in fractions]
    fig.add_trace(go.Scatter(x=fractions, y=detectable, mode="lines", name=row.stratum,
                             line={"color": h.STRATUM_COLOR[stratum], "width": 2.6}))
fig.add_hline(y=abs(h.intent_to_treat_contrast(40.0, "age_51_plus")),
              line={"color": "#111", "dash": "dash"},
              annotation_text="true 40 mg harm in the 51+ band")
for t in REVIEWS:
    fig.add_vline(x=t, line={"color": "rgba(209,72,63,0.35)", "dash": "dot"})
fig.add_vline(x=REVIEWS[0], line={"color": "#d1483f", "dash": "dot"},
              annotation_text="first review")
fig.update_yaxes(range=[0, 12])
fig

In [ ]:
oldest = strata_plan.iloc[2]
harm = abs(h.intent_to_treat_contrast(40.0, "age_51_plus"))
print(f"the 51+ band's 40 mg contrast at each scheduled safety review")
rows = []
for t in REVIEWS:
    se_t = float(oldest.se) / float(np.sqrt(t))
    rows.append([f"{t:.3f}", f"{se_t:.2f}", f"{coefficient_mde(se_t, power=TARGET_POWER).effect:.2f}",
                 f"{coefficient_power(harm, se_t).power:.3f}"])
table(rows, headers=("information", "se", "MDE(80%)", "power at the true harm"))
print(f"\nAt the first review the test has {coefficient_power(harm, float(oldest.se) / np.sqrt(REVIEWS[0])).power:.0%} power against the harm that is")
print(f"actually happening, and its MDE of "
      f"{coefficient_mde(float(oldest.se) / float(np.sqrt(REVIEWS[0])), power=TARGET_POWER).effect:.1f} "
      f"{h.OUTCOME_UNIT} sits *above* it. A single look is a coin flip.")

And a single look is not the problem. Six of them are. Testing at 5 % on every review
and stopping the first time it clears has no error control at all — the trial gets six
chances, and across the twelve monitored contrasts it gets seventy-two. Notebook 5
replaces the repeated test with a boundary whose false-stop rate is a computed number,
and prices exactly what that costs in sensitivity.
## 6. If randomization had been by clinic instead of by unit

HYPER-3 randomizes individual units, which is why nothing above carries a design
effect. Had the trial randomized **clinics** — sometimes the only option when the
intervention is a care pathway rather than a tablet — units inside a clinic would be
correlated and the effective sample size would collapse. `design_effect` is
`1 + (m − 1)·ICC`; at 25 units per clinic, an ICC of 0.03 already costs a third of the
trial.

In [ ]:
clinic = Unit(name="clinic", dimension=D.entity, kind="cluster")
iccs = np.linspace(0.0, 0.10, 51)
fig = h.figure("Design effect: what clustering would have cost", "intra-clinic correlation",
               "effective sample size as a share of n", height=380)
for size, colour in ((10, "#8ab4e8"), (25, "#2f7fd1"), (50, "#20456b")):
    shares = [effective_sample_size(16, size, float(icc)) / (16 * size) for icc in iccs]
    fig.add_trace(go.Scatter(x=iccs, y=shares, mode="lines", name=f"{size} units per clinic",
                             line={"color": colour, "width": 2.6}))
fig.add_vline(x=0.03, line={"color": "#d1483f", "dash": "dot"}, annotation_text="ICC 0.03")
print("design effect at 25 units per clinic, ICC 0.03:", round(design_effect(25, 0.03), 3))
fig

In [ ]:
clustered = ClusterDesign(unit=clinic, n_clusters=16, cluster_size=25, icc=0.03, allocation=0.5)
cluster_result = cluster_power(clustered, MEANINGFUL, SD_WINDOW, alpha=ALPHA)
print(f"16 clinics x 25 units = {clustered.n_clusters * clustered.cluster_size} units, "
      f"but effective n = {effective_sample_size(16, 25, 0.03):.0f}")
print(f"power for {MEANINGFUL} {h.OUTCOME_UNIT}: {cluster_result.power:.3f} "
      f"(individually randomized: {power(400, MEANINGFUL, SD_WINDOW).power:.3f})")
print(f"cluster MDE: {cluster_mde(clustered, SD_WINDOW, power=TARGET_POWER).effect:.2f} {h.OUTCOME_UNIT}")
needed_clusters = clusters_needed(MEANINGFUL, SD_WINDOW, cluster_size=25, icc=0.03, power=TARGET_POWER)
print("clinics needed:", needed_clusters.n if isinstance(needed_clusters, SampleSize) else needed_clusters)

tradeoff = holdout_tradeoff(clustered, SD_WINDOW, (0.1, 0.2, 0.3, 0.4, 0.5))
assert isinstance(tradeoff, HoldoutTradeoff)
fig = h.figure("Control share in a clinic-randomized version of the trial",
               "share of clinics held as control", f"MDE ({h.OUTCOME_UNIT})", height=360)
fig.add_trace(go.Scatter(x=tradeoff.fractions, y=tradeoff.mdes, mode="lines+markers",
                         line={"color": "#2f7fd1", "width": 2.6}, showlegend=False))
fig.add_vline(x=tradeoff.best_fraction, line={"color": "#d1483f", "dash": "dot"},
              annotation_text=f"best {tradeoff.best_fraction:.0%}")
print("best control share:", tradeoff.best_fraction, "| control clinics there:",
      tradeoff.n_holdout[tradeoff.fractions.index(tradeoff.best_fraction)])
fig

`match_clusters` pairs clinics on their pre-period pressure and randomizes one of each
pair, which is what a clinic-randomized version of HYPER-3 would do instead of
stratifying units by age.

In [ ]:
rng = np.random.default_rng(3)
pre_period = rng.normal(143.0, 5.0, size=(16, 6)) + rng.normal(0.0, 1.0, size=(16, 1))
assignment = match_clusters(pre_period, labels=[f"clinic_{i:02d}" for i in range(16)],
                            unit=clinic, metric="trajectory", seed=7)
assert isinstance(assignment, Assignment)
print(f"{len(assignment.pairs)} matched pairs, {len(assignment.unpaired)} unpaired")
print(f"pre-period means: treated {assignment.pre_mean_treated:.2f}, "
      f"control {assignment.pre_mean_control:.2f}, standardized gap {assignment.pre_smd:+.3f}")
print("mean within-pair distance:", round(assignment.mean_pair_distance, 3))

## 7. What a millimetre of mercury costs

A trial has a budget, and the question a payer asks is not "is it significant" but
"what is the most this could plausibly cost per mmHg". `cost_per_outcome_interval`
gives the **Fieller** interval for `cost / effect`, which is the honest one: when the
effect's interval covers zero the ratio's upper bound is infinite, and the function
says so rather than returning a number.

In [ ]:
COST = 2_400_000.0  # what running HYPER-3 costs, in the trial's numeraire
rows = []
for arm in h.ARMS[1:]:
    estimate = ols(h.contrast_frame(final, arm), "change", "treated", h.ancova_covariates(None))
    ratio = cost_per_outcome_interval(COST, -estimate.estimate, estimate.se, alpha=ALPHA)
    assert isinstance(ratio, CostPerOutcomeInterval)
    upper = "unbounded" if ratio.upper is None else f"{ratio.upper:,.0f}"
    rows.append(
        [h.ARM_LABEL[arm], f"{-estimate.estimate:+.2f} ± {estimate.se:.2f}",
         f"{ratio.estimate:,.0f}", f"[{ratio.lower:,.0f}, {upper}]",
         f"{ratio.method}, {ratio.status}"]
    )
table(rows, headers=("arm", "reduction", f"cost per {h.OUTCOME_UNIT}", "interval", "method"))

power_result = cost_per_outcome_power(COST, MEANINGFUL, float(plan["se"].mean()),
                                      threshold=800_000.0)
assert isinstance(power_result, CostPerOutcomePower)
print(f"\nprobability the design bounds cost per {h.OUTCOME_UNIT} below 800,000: "
      f"{power_result.power:.3f} (needs an effect of at least "
      f"{power_result.effect_required:.2f} {h.OUTCOME_UNIT})")
print(f"largest cost per {h.OUTCOME_UNIT} this design can bound at all: "
      f"{max_detectable_cost_per_outcome(COST, pooled_mde):,.0f}")

In [ ]:
fig = h.figure("Cost per mmHg the design can bound, against the effect it turns out to have",
               f"true reduction ({h.OUTCOME_UNIT})", "cost per mmHg", height=380)
effects = np.linspace(1.0, 8.0, 71)
for label, se, colour in (("pooled contrast", float(plan["se"].mean()), "#2f7fd1"),
                          ("inside the 51+ stratum",
                           float(strata_plan.loc[2, "se"]), "#b5453b")):
    bounds = []
    for effect in effects:
        interval = cost_per_outcome_interval(COST, float(effect), se, alpha=ALPHA)
        bounds.append(interval.upper if interval.upper is not None else np.nan)
    fig.add_trace(go.Scatter(x=effects, y=bounds, mode="lines", name=label,
                             line={"color": colour, "width": 2.6}))
fig.update_yaxes(type="log")
fig.add_vline(x=MEANINGFUL, line={"color": "#111", "dash": "dot"},
              annotation_text=f"{MEANINGFUL} {h.OUTCOME_UNIT}")
fig

## What this notebook decided

- If HYPER-3 only had to answer the pooled question, 71 units would do it. It enrolls
  400 because the binding constraint is elsewhere: three doses share one control, and
  the protocol pre-specifies safety monitoring **inside** each age stratum.
- 400 units in 2 : 1 : 1 : 1 blocks. The square-root rule wants 1.73 : 1; 2 : 1 costs a
  quarter of a percent of standard error and gives whole-numbered blocks, so the
  protocol takes it.
- Averaging four weekly readings instead of reading one visit halves the endpoint
  variance and is worth roughly 80 units of enrollment. It is the largest design lever
  in the trial and it is free — the visits were already scheduled.
- At closeout the trial detects about 2.4 mmHg pooled and 3.8–4.8 mmHg inside a
  stratum, so a 5.8 mmHg stratum-level harm is within reach **at the end**. At the first
  safety review it is not: 69 % power, and a minimum detectable effect of 6.6 mmHg
  sitting above the harm that is actually happening. And a rule that simply tests at 5 %
  on each of six reviews, across twelve contrasts, controls nothing at all. Notebook 5
  replaces it with a boundary whose false-stop rate is computed rather than assumed.
- Had the trial randomized clinics rather than units, an ICC of 0.03 at 25 units per
  clinic would have cost a third of the effective sample. Individual randomization is
  worth defending.
- At this size the trial bounds the cost per mmHg for an arm that works and returns an
  unbounded upper limit for one that does not. The Fieller interval says which case it
  is in rather than quietly reporting a ratio.